# Instabilities and waves

`takeout.analysis` reads model output without the code that produced it. This example builds a synthetic *instability record* of electrons drifting through ions, lets `instability_report` find, name and check the growing mode, and compares the ion-acoustic root of `takeout.analysis.ion_acoustic` with the fluid sound wave. `instability_report` needs plasma-plots: install `takeout[analysis]`.

In [ ]:
import numpy as np
import plasma_plots
import xarray as xr
from scipy import constants

from takeout.analysis.instabilities import instability_report
from takeout.analysis.ion_acoustic import ion_acoustic_root

plasma_plots.set_backend("plotly")
plasma_plots.set_math("latex");

## An instability record

picnic's `InstabilityMonitor` records this layout during a run: the potential along each periodic axis (`potential_x`), the energies, the mean density, drift and temperature of each species, a velocity sample at the first and the latest record, and the geometry in the attributes. Here it is made up: electrons drifting at 25 thermal speeds through cold protons, with the potential's mode 3 growing out of the noise at $\gamma = 0.05\,\omega_{pe}$ and travelling at $\omega_r = 0.03\,\omega_{pe}$.

In [ ]:
rng = np.random.default_rng(0)
density, drift, v_t = 1e15, 5e6, 2e5
omega_pe = np.sqrt(density * constants.e**2 / (constants.epsilon_0 * constants.m_e))
nodes, length = 128, 3 * 2 * np.pi * drift / omega_pe
t = np.linspace(0.0, 600.0 / omega_pe, 600)
x = np.arange(nodes) * length / nodes
k = 2 * np.pi * 3 / length
amplitude = 1e-7 * np.exp(0.05 * omega_pe * t)
potential = 1e-6 * rng.standard_normal((t.size, nodes)) + (amplitude / (1 + amplitude))[
    :, None
] * np.cos(k * x[None] - 0.03 * omega_pe * t[:, None])

count = 20_000
mass = [constants.m_e, constants.m_p]
drifts = np.array([[drift, 0.0, 0.0], [0.0, 0.0, 0.0]])
temperatures = np.array([[constants.m_e * v_t**2] * 3, [constants.m_p * 1e6] * 3])
velocities = np.stack(
    [drift + v_t * rng.standard_normal(count), 1e3 * rng.standard_normal(count)]
)


def over_time(values):
    return np.repeat(np.asarray(values)[None], t.size, axis=0)


record = xr.Dataset(
    {
        "potential_x": (("t", "x"), potential, {"units": "V"}),
        "field_energy": ("t", np.full(t.size, 1e-20), {"units": "J"}),
        "kinetic_energy": (
            ("t", "species"),
            over_time(
                0.5
                * density
                * (temperatures.sum(-1) + np.array(mass) * drifts[:, 0] ** 2)
                * length
            ),
            {"units": "J"},
        ),
        "density": (("t", "species"), over_time([density, density]), {"units": "m^-3"}),
        "drift": (("t", "species", "component"), over_time(drifts), {"units": "m/s"}),
        "temperature": (
            ("t", "species", "component"),
            over_time(temperatures),
            {"units": "J"},
        ),
        "sample_velocity": (
            ("snapshot", "species", "component", "sample"),
            np.repeat(np.repeat(velocities[None, :, None], 3, axis=2), 2, axis=0),
            {"units": "m/s"},
        ),
        "sample_weight": (("snapshot", "species", "sample"), np.ones((2, 2, count))),
    },
    coords={
        "t": ("t", t, {"units": "s"}),
        "x": ("x", x, {"units": "m"}),
        "species": ["electrons", "ions"],
        "component": ["x", "y", "z"],
        "snapshot": [t[0], t[-1]],
    },
    attrs={
        "ndim": 1,
        "nvel": 3,
        "periodic": [1],
        "lengths": [length],
        "spacing": [length / nodes],
        "dt": 0.05 / omega_pe,
        "mass": mass,
        "charge": [-constants.e, constants.e],
        "magnetic_field": 0.0,
        "magnetic_field_direction": [0.0, 0.0, 0.0],
    },
)
record

## The report

`instability_report` searches every Fourier mode for an exponential phase out of the noise, takes the growth rate from the fit and the frequency from the phase, names the mode from the plasma's dimensionless numbers, checks the velocity distributions with the Penrose criterion and solves the electrostatic dispersion relation of the measured populations at each $k$:

In [ ]:
report = instability_report(record)
print(report)
mode = report.modes[0]
print(f"\n{mode.name}: {mode.reason}")
print(f"gamma / omega_pe = {mode.growth_rate / omega_pe:.4f} (0.05 put in)")

The made-up grid is coarser than the Debye length, which the report flags as a numerical risk. The amplitude of mode 3 against time, with the growth phase the report found (the noise floor, the exponential phase and the saturation):

In [ ]:
coefficient = np.abs(np.fft.rfft(record["potential_x"].values, axis=1)[:, 3]) / nodes
time = xr.DataArray(t * omega_pe, dims="t", attrs={"long_name": "$\\omega_{pe} t$"})
amplitude = xr.DataArray(
    coefficient,
    coords={"t": time},
    dims="t",
    attrs={"long_name": "$|\\hat\\phi_3|$", "units": "V"},
)
fit = xr.DataArray(
    coefficient[np.argmin(np.abs(t - mode.start))]
    * np.exp(mode.growth_rate * (t - mode.start)),
    coords={"t": time},
    dims="t",
    attrs={"long_name": "fit", "units": "V"},
).where((t >= mode.start) & (t <= mode.stop))
curves = xr.concat(
    [amplitude, fit], dim=xr.Variable("curve", ["mode 3", "fit"])
).assign_attrs(amplitude.attrs)
with plasma_plots.figure() as fig:
    curves.plasma.plot.profiles(x="t", over="curve", ax=fig[0])
    fig[0].set_yscale("log")

## The ion-acoustic wave

`ion_acoustic_root` solves the kinetic dispersion relation of Maxwellian electrons and ions (frequencies in $\omega_{pe}$, wavenumbers in $1/\lambda_{De}$). With $T_e/T_i = 20$ and $m_i/m_e = 100$ its real part follows the fluid sound wave $k c_s / \sqrt{1 + k^2\lambda_{De}^2}$ with the ion correction $\sqrt{1 + 3 T_i/T_e}$, and the Landau damping grows with $k$:

In [ ]:
wavenumbers = np.linspace(0.02, 1.0, 50)
roots = np.array(
    [
        ion_acoustic_root(q, temperature_ratio=20.0, mass_ratio=100.0)
        for q in wavenumbers
    ]
)
k_axis = xr.DataArray(wavenumbers, dims="k", attrs={"long_name": "$k \\lambda_{De}$"})
fluid = wavenumbers / np.sqrt(100.0) * np.sqrt(1 + 3 / 20) / np.sqrt(1 + wavenumbers**2)
branches = xr.DataArray(
    np.stack([roots.real, fluid, -roots.imag]),
    coords={
        "curve": ["kinetic $\\omega_r$", "fluid $\\omega_r$", "damping $-\\gamma$"],
        "k": k_axis,
    },
    dims=("curve", "k"),
    attrs={"long_name": "$\\omega / \\omega_{pe}$"},
)
with plasma_plots.figure() as fig:
    branches.plasma.plot.profiles(x="k", over="curve", ax=fig[0])